<a href="https://colab.research.google.com/github/bahaali273/COMP9170-Hospital-Readmission/blob/bahaa-dev/notebooks/03_kumo_tabular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# NVIDIA Kumo-Tabular — Hospital Readmission Prediction

**Course:** COMP 9170 — Data Science  
**Author:** Bahaa Al-Juboori

## 1. Environment and Compatibility Check

### Decisions
- Investigate NVIDIA Kumo-Tabular for binary classification.
- Check the available GPU and software environment.
- Reuse the team's existing training and test partitions.
- Investigate compatibility with 133 input features.
- Keep the original test set reserved for final evaluation.


### Results
- Python version: 3.13.16.
- PyTorch version: 2.14.1+cu130.
- CUDA available: True.
- GPU: NVIDIA Tesla T4.
- GPU memory: 14.56 GB.
- The GPU environment was configured successfully.
- Kumo-Tabular inference has not yet been tested.



In [1]:

import sys
import torch

print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("No GPU detected.")


Python version: 3.13.16
PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56



## 2. NVIDIA Kumo-Tabular Installation

### Decisions
- Use NVIDIA's official structured-data-models package.
- Install the library in the GPU-enabled Colab environment.
- Verify that Kumo-Tabular can be imported successfully.
- Do not load model weights or run predictions yet.


### Results
- NVIDIA structured-data-models version: 0.2.1.
- The SDM package was installed successfully.
- The `sdm` library imported without errors.
- The KumoTabular class was detected successfully.
- Model weights have not yet been downloaded or loaded.



In [2]:

%pip install -q structured-data-models

import importlib.metadata

print("Installed SDM version:",
      importlib.metadata.version("structured-data-models"))

import sdm

print("SDM import successful!")
print("Kumo-Tabular available:",
      hasattr(sdm.models, "KumoTabular"))


Installed SDM version: 0.2.1
SDM import successful!
Kumo-Tabular available: True



## 3. Dataset Compatibility Check

### Decisions
- Reuse the processed training dataset from the shared pipeline.
- Verify that all input features are numeric and contain no missing values.
- Inspect the binary target and feature count.
- Avoid using the held-out test set during initial compatibility testing.
- Start with a small validation experiment before full-scale inference.

### Results

### Results
- Training dataset: 55,976 rows and 134 columns.
- Number of input features: 133.
- Target classes: 0 and 1.
- All input features are numeric.
- No missing values were detected.
- Positive-class prevalence: 8.97%.
- Dataset memory usage: 57.23 MB.
- The dataset passed initial compatibility checks for Kumo-Tabular.
- Model inference and GPU memory compatibility have not yet been tested.



In [4]:

import pandas as pd
import numpy as np

train_df = pd.read_csv("/content/train_clean.csv")

target = "readmit_30"
X = train_df.drop(columns=[target])
y = train_df[target]

print("Training shape:", train_df.shape)
print("Number of features:", X.shape[1])
print("Target classes:", sorted(y.unique().tolist()))
print("Non-numeric features:", X.select_dtypes(exclude="number").columns.tolist())
print("Missing values:", int(train_df.isna().sum().sum()))
print("Target prevalence:", round(y.mean(), 4))
print("Memory usage (MB):",
      round(train_df.memory_usage(deep=True).sum() / 1024**2, 2))


Training shape: (55976, 134)
Number of features: 133
Target classes: [0, 1]
Non-numeric features: []
Missing values: 0
Target prevalence: 0.0897
Memory usage (MB): 57.23



## 4. Small In-Context Dataset Preparation

### Decisions
- Use the existing cleaned training dataset.
- Create a stratified development subset with 500 context rows and 100 query rows.
- Keep the held-out test dataset untouched.
- Convert the data to NVIDIA's TableTensor format.
- Mark the binary target as categorical.
- Verify successful conversion before loading model weights.


### Results
- Selected 600 samples from the processed training dataset.
- Context dataset: 500 samples.
- Query dataset: 100 samples.
- Input features: 133.
- Context positive-class prevalence: 9.0%.
- Query positive-class prevalence: 9.0%.
- Device: CUDA GPU.
- Successfully inferred column types using `sdm.infer_stypes()`.
- Successfully converted the dataset to NVIDIA TableTensor format.
- No model inference has been performed yet.
- The held-out test dataset remains untouched.



In [5]:

import torch
import sdm
from sklearn.model_selection import train_test_split

SEED = 42
TARGET = "readmit_30"

# Create a stratified development sample
dev_df, _ = train_test_split(
    train_df,
    train_size=600,
    stratify=train_df[TARGET],
    random_state=SEED
)

# Separate context rows from query rows
context_df, query_df = train_test_split(
    dev_df,
    train_size=500,
    stratify=dev_df[TARGET],
    random_state=SEED
)

# Recombine for consistent TableTensor conversion
sample_df = pd.concat(
    [context_df, query_df],
    ignore_index=True
)

# NVIDIA Kumo-Tabular input format
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

stypes = sdm.infer_stypes(
    sample_df,
    overrides={TARGET: "categorical"}
)

table = sdm.TableTensor.from_pandas(
    df=sample_df,
    stypes=stypes,
    device=device
)

print("Context rows:", len(context_df))
print("Query rows:", len(query_df))
print("Number of features:", sample_df.shape[1] - 1)
print("Context positive rate:",
      round(context_df[TARGET].mean(), 4))
print("Query positive rate:",
      round(query_df[TARGET].mean(), 4))
print("Device:", device)
print("TableTensor conversion successful!")


Context rows: 500
Query rows: 100
Number of features: 133
Context positive rate: 0.09
Query positive rate: 0.09
Device: cuda
TableTensor conversion successful!
